# Case Study-1: Tokenizing E-commerce Customer Reviews (NLTK)

**Dataset:** [Women's E-Commerce Clothing Reviews](https://www.kaggle.com/datasets/nicapotato/womens-ecommerce-clothing-reviews) (Kaggle). Text column: `Review Text`, plus `Rating` (1-5).

If the CSV isn't found, the notebook falls back to the 3 sample reviews from the case study so every cell still runs.

In [6]:
import nltk
import pandas as pd
from nltk.tokenize import sent_tokenize, word_tokenize

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

True

In [7]:
import os

candidates = [
    'Womens Clothing E-Commerce Reviews.csv',
    '/kaggle/input/womens-ecommerce-clothing-reviews/Womens Clothing E-Commerce Reviews.csv',
]
path = next((p for p in candidates if os.path.exists(p)), None)

if path:
    df = pd.read_csv(path, index_col=0)
    df = df[['Review Text', 'Rating']].dropna(subset=['Review Text']).reset_index(drop=True)
    df = df.rename(columns={'Review Text': 'review'})
    df = df.sample(500, random_state=42).reset_index(drop=True)   # work on a 500-review sample
    print('Loaded Kaggle data:', df.shape)
else:
    print('Kaggle CSV not found -> using the 3 case-study sample reviews')
    df = pd.DataFrame({
        'review': [
            "The product quality is excellent! I am very satisfied with the purchase. Will definitely recommend it.",
            "Terrible customer service. I had to wait two weeks for a response. Not buying again!",
            "Average product. Delivery was on time, but the packaging was damaged.",
        ],
        'Rating': [5, 1, 3],
    })
df.head()

Loaded Kaggle data: (500, 2)


,review,Rating
0,This sweater is so beautiful on. it is thick m...,5
1,This piece is almost what i want... i tried on...,4
2,Really like this blouse but am returning for a...,4
3,These are the perfect light weight relaxing su...,5
4,These look nothing like the picture! they are ...,2


In [8]:
def tokenize_review(text):
    """Return list of sentences; each sentence is a list of word tokens."""
    sentences = sent_tokenize(text)
    return sentences, [word_tokenize(s) for s in sentences]

df['sentences'], df['words'] = zip(*df['review'].apply(tokenize_review))

# Show the first 3 reviews in detail
for idx in range(min(3, len(df))):
    print(f"Review {idx+1}: {df.loc[idx, 'review']}")
    for i, (s, w) in enumerate(zip(df.loc[idx, 'sentences'], df.loc[idx, 'words']), 1):
        print(f"  Sentence {i}: {s}")
        print(f"  Words    {i}: {w}")
    print()

Review 1: This sweater is so beautiful on. it is thick material, but does not make you look boxy. it fits so nicely and is flattering. the design is just gorgeous. if you are considering this sweater-- get it, you won't regret it!

i'm normally a small (4-6) and i ordered a small. it fits true to size. the fit is *perfect* so if you want it slightly more relaxed, order one size up.
  Sentence 1: This sweater is so beautiful on.
  Words    1: ['This', 'sweater', 'is', 'so', 'beautiful', 'on', '.']
  Sentence 2: it is thick material, but does not make you look boxy.
  Words    2: ['it', 'is', 'thick', 'material', ',', 'but', 'does', 'not', 'make', 'you', 'look', 'boxy', '.']
  Sentence 3: it fits so nicely and is flattering.
  Words    3: ['it', 'fits', 'so', 'nicely', 'and', 'is', 'flattering', '.']
  Sentence 4: the design is just gorgeous.
  Words    4: ['the', 'design', 'is', 'just', 'gorgeous', '.']
  Sentence 5: if you are considering this sweater-- get it, you won't regret it!
  W

In [9]:
df['num_sentences'] = df['sentences'].apply(len)
df['words_per_sentence'] = df['words'].apply(lambda ws: [len(w) for w in ws])

print(df[['num_sentences', 'words_per_sentence']].head(10))
print('\nAverage sentences per review:', round(df['num_sentences'].mean(), 2))
print('Sentence-count distribution:')
print(df['num_sentences'].value_counts().sort_index().head(10))

   num_sentences             words_per_sentence
0              8   [7, 13, 8, 6, 16, 14, 6, 20]
1              6        [38, 19, 13, 23, 16, 3]
2              2                       [12, 19]
3              5           [10, 25, 38, 27, 11]
4              7      [7, 13, 20, 20, 11, 8, 5]
5              8  [31, 17, 7, 13, 11, 15, 7, 4]
6              4                [11, 6, 19, 20]
7              5            [4, 16, 10, 14, 14]
8              4                [39, 19, 36, 3]
9              7        [5, 9, 25, 6, 29, 8, 5]

Average sentences per review: 4.75
Sentence-count distribution:
num_sentences
1      15
2      52
3      75
4     108
5      79
6      73
7      51
8      28
9      12
10      3
Name: count, dtype: int64


In [10]:
POSITIVE = {'excellent','great','good','love','loved','perfect','satisfied','recommend','amazing',
            'beautiful','comfortable','soft','nice','happy','wonderful','best','gorgeous','flattering',
            'cute','lovely','pretty','fantastic','awesome','favorite','fits'}
NEGATIVE = {'terrible','bad','poor','awful','disappointed','disappointing','damaged','worst','cheap',
            'horrible','uncomfortable','ugly','small','tight','returned','return','waste','wait',
            'itchy','scratchy','flimsy','broke','wrong','unflattering'}

def find_sentiment_words(word_lists):
    tokens = [t.lower() for sent in word_lists for t in sent if t.isalpha()]
    return ([t for t in tokens if t in POSITIVE],
            [t for t in tokens if t in NEGATIVE])

df['pos_words'], df['neg_words'] = zip(*df['words'].apply(find_sentiment_words))
df['pos_count'] = df['pos_words'].apply(len)
df['neg_count'] = df['neg_words'].apply(len)

df[['review', 'pos_words', 'neg_words']].head(5)

,review,pos_words,neg_words
0,This sweater is so beautiful on. it is thick m...,"[beautiful, fits, flattering, gorgeous, fits, ...","[small, small]"
1,This piece is almost what i want... i tried on...,"[pretty, love, nice, nice]",[]
2,Really like this blouse but am returning for a...,[],"[tight, tight]"
3,These are the perfect light weight relaxing su...,"[perfect, soft, perfect, perfect]",[]
4,These look nothing like the picture! they are ...,[],[unflattering]


In [11]:
from collections import Counter

print('Top positive words:', Counter(w for ws in df['pos_words'] for w in ws).most_common(10))
print('Top negative words:', Counter(w for ws in df['neg_words'] for w in ws).most_common(10))

# Simple label + sanity check against the star rating
df['lexicon_label'] = df.apply(
    lambda r: 'positive' if r.pos_count > r.neg_count else ('negative' if r.neg_count > r.pos_count else 'neutral'),
    axis=1)
print('\nAverage star rating by lexicon label:')
print(df.groupby('lexicon_label')['Rating'].agg(['count', 'mean']).round(2))

Top positive words: [('love', 195), ('great', 126), ('perfect', 81), ('flattering', 78), ('nice', 68), ('cute', 67), ('soft', 60), ('comfortable', 60), ('pretty', 59), ('fits', 56)]
Top negative words: [('small', 79), ('tight', 24), ('return', 13), ('disappointed', 12), ('returned', 11), ('wait', 11), ('unflattering', 9), ('itchy', 7), ('cheap', 7), ('bad', 6)]

Average star rating by lexicon label:
               count  mean
lexicon_label             
negative          36  3.14
neutral           79  3.57
positive         385  4.36


In [12]:
def flag_long_reviews(dataframe, threshold=3):
    """Return reviews with more than `threshold` sentences, flagged for further analysis."""
    flagged = dataframe[dataframe['num_sentences'] > threshold].copy()
    flagged['flag'] = 'REVIEW FURTHER'
    return flagged[['review', 'num_sentences', 'flag']]

flagged = flag_long_reviews(df)
print(f'{len(flagged)} of {len(df)} reviews flagged (> 3 sentences)')
flagged.head()

358 of 500 reviews flagged (> 3 sentences)


,review,num_sentences,flag
0,This sweater is so beautiful on. it is thick m...,8,REVIEW FURTHER
1,This piece is almost what i want... i tried on...,6,REVIEW FURTHER
3,These are the perfect light weight relaxing su...,5,REVIEW FURTHER
4,These look nothing like the picture! they are ...,7,REVIEW FURTHER
5,I ordered this shirt and was happy to see that...,8,REVIEW FURTHER
